# Enkrypt AI — Anti-Hallucination Detectors

This notebook walks you through Enkrypt AI's two anti-hallucination APIs:

| Detector | What it checks | API endpoint |
|---|---|---|
| **Adherence** | Is the LLM answer grounded in the provided context? | `/guardrails/adherence` |
| **Relevancy** | Does the LLM answer actually address the question? | `/guardrails/relevancy` |

Run each cell top-to-bottom. You only need to set your API key in the **Setup** cell.

## Prerequisites

```bash
pip install requests python-dotenv openai
```

You'll also need:
- An **Enkrypt AI** API key — get one free at **https://app.enkryptai.com**
- An **OpenAI** API key — required for Part 5's summary correction step (**https://platform.openai.com**)

In [1]:
import os
import json
import requests
from concurrent.futures import ThreadPoolExecutor

# ── Set your API key ──────────────────────────────────────────────────────
# Option A: set it directly here (fine for notebooks, never commit it)
# Option B: load from a .env file in the same folder
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY", "YOUR_API_KEY_HERE")

ADHERENCE_URL = "https://api.enkryptai.com/guardrails/adherence"
RELEVANCY_URL = "https://api.enkryptai.com/guardrails/relevancy"

HEADERS = {
    "apikey": ENKRYPTAI_API_KEY,
    "Content-Type": "application/json",
}

print("Setup complete. API key set:", bool(ENKRYPTAI_API_KEY and ENKRYPTAI_API_KEY != "YOUR_API_KEY_HERE"))

Setup complete. API key set: True


---
## Part 1 — Adherence Detector

**Use case:** RAG systems, document Q&A, knowledge bases.

The adherence detector:
1. Extracts **atomic facts** from the LLM answer
2. Checks each fact against the provided context
3. Returns a score from **0** (fully hallucinated) to **1** (fully grounded)

Fact scores: `2` = supported, `0` = unsupported / incorrect

In [2]:
def check_adherence(llm_answer: str, context: str) -> dict:
    payload = {"llm_answer": llm_answer, "context": context}
    response = requests.post(ADHERENCE_URL, json=payload, headers=HEADERS)
    response.raise_for_status()
    return response.json()


def _parse_nested(value):
    """The API may return nested objects as a JSON string — parse them safely."""
    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            return {}
    return value if isinstance(value, dict) else {}


# ANSI color helpers
# Color key (adherence):
#   Green  — TYPE 2: fact is directly and explicitly supported by the context
#   Yellow — TYPE 1: fact is inferred / not directly stated in the context
#   Red    — TYPE 0: fact is incorrect and contradicts the context
def _c(text: str, color: str) -> str:
    codes = {"green": "\033[92m", "yellow": "\033[93m", "red": "\033[91m", "bold": "\033[1m", "reset": "\033[0m"}
    return f"{codes.get(color, '')}{text}{codes['reset']}"


def _fact_color(adherence_score: int, reasoning: str) -> tuple[str, str, str]:
    """Return (color, type_label, status_label) for a single fact."""
    if adherence_score == 2:
        return "green", "TYPE 2", "SUPPORTED  "
    if "TYPE 1" in reasoning:
        return "yellow", "TYPE 1", "INFERRED   "
    return "red", "TYPE 0", "INCORRECT  "


def display_adherence(result: dict) -> None:
    score = result.get("summary", {}).get("adherence_score", 0)
    details = result.get("details", {})
    atomic_facts = details.get("atomic_facts", [])
    adherence_list = details.get("adherence_list", [])
    adherence_response = _parse_nested(details.get("adherence_response", {}))
    chain_of_thought = adherence_response.get("chain_of_thought", [])

    # Pad chain_of_thought in case it's shorter than atomic_facts
    cot_padded = list(chain_of_thought) + [""] * max(0, len(atomic_facts) - len(chain_of_thought))

    status = "PASS" if score >= 0.5 else ("WARN" if score >= 0.4 else "FAIL")
    score_color = "green" if score >= 0.5 else ("yellow" if score >= 0.4 else "red")
    print(f"Adherence Score : {_c(f'{score:.2f} ({score*100:.1f}%)  [{status}]', score_color)}")

    if atomic_facts:
        print(f"\nAtomic Facts ({len(atomic_facts)}):")
        for i, (fact, fs, reasoning) in enumerate(zip(atomic_facts, adherence_list, cot_padded), 1):
            color, type_label, status_label = _fact_color(fs, reasoning)
            print(f"  {i}. {_c(f'[{type_label} — {status_label}]', color)} {fact}")

    if chain_of_thought:
        print("\nReasoning:")
        for i, r in enumerate(chain_of_thought, 1):
            color = "green" if r.startswith("TYPE 2") else ("yellow" if r.startswith("TYPE 1") else "red")
            print(f"  {i}. {_c(r, color)}")

In [3]:
# Example 1a: Answer that is GROUNDED in context
context = """
Indian scientists have made significant contributions to various scientific fields.
C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering.
Other notable scientists include Srinivasa Ramanujan, a mathematical genius.
"""

llm_answer_good = "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering."

print("Context   :", context.strip()[:100], "...")
print("LLM Answer:", llm_answer_good)
print()

result = check_adherence(llm_answer_good, context)
display_adherence(result)

Context   : Indian scientists have made significant contributions to various scientific fields.
C.V. Raman won t ...
LLM Answer: C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering.

Adherence Score : 1.00 (100.0%)  [PASS]

Atomic Facts (2):
  1. [TYPE 2 — SUPPORTED  ] C.V. Raman won the Nobel Prize for Physics in 1930.
  2. [TYPE 2 — SUPPORTED  ] C.V. Raman won the Nobel Prize for his work on light scattering.

Reasoning:
  1. TYPE 2: This fact is directly supported by the context, which states that C.V. Raman won the Nobel Prize for Physics in 1930.
  2. TYPE 2: This fact is directly supported by the context, which states that C.V. Raman won the Nobel Prize for his work on light scattering.


In [4]:
# Example 1b: Answer with HALLUCINATED facts
llm_answer_bad = (
    "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on "
    "making some science stuff or cooking stuff."
)

print("LLM Answer:", llm_answer_bad)
print()

result = check_adherence(llm_answer_bad, context)
display_adherence(result)

LLM Answer: C.V. Raman won the Nobel Prize for Physics in 1930 for his work on making some science stuff or cooking stuff.

Adherence Score : 0.50 (50.0%)  [PASS]

Atomic Facts (2):
  1. [TYPE 2 — SUPPORTED  ] C.V. Raman won the Nobel Prize for Physics in 1930.
  2. [TYPE 0 — INCORRECT  ] C.V. Raman won the Nobel Prize for his work on making some science stuff or cooking stuff.

Reasoning:
  1. TYPE 2: This fact is directly supported by the context, which states that C.V. Raman won the Nobel Prize for Physics in 1930.
  2. TYPE 0: The context specifies that C.V. Raman won the Nobel Prize for his work on light scattering, not for 'making some science stuff or cooking stuff'. CORRECTED_FACT: C.V. Raman won the Nobel Prize for his work on light scattering.


In [5]:
# Example 1c: Answer completely outside the context
context_policy = (
    "Our refund policy allows customers to return items within 30 days of purchase "
    "for a full refund. Items must be unused and in original packaging. "
    "Electronics are excluded from this policy."
)

llm_answer_off = (
    "You can return any item within 90 days for a full refund, "
    "including all electronics and opened packages."
)

print("Context   :", context_policy)
print("LLM Answer:", llm_answer_off)
print()

result = check_adherence(llm_answer_off, context_policy)
display_adherence(result)

Context   : Our refund policy allows customers to return items within 30 days of purchase for a full refund. Items must be unused and in original packaging. Electronics are excluded from this policy.
LLM Answer: You can return any item within 90 days for a full refund, including all electronics and opened packages.

Adherence Score : 0.00 (0.0%)  [FAIL]

Atomic Facts (3):
  1. [TYPE 0 — INCORRECT  ] You can return any item within 90 days for a full refund.
  2. [TYPE 0 — INCORRECT  ] You can return all electronics within 90 days for a full refund.
  3. [TYPE 0 — INCORRECT  ] You can return opened packages within 90 days for a full refund.

Reasoning:
  1. TYPE 0: The context states that items can be returned within 30 days, not 90 days, for a full refund. CORRECTED_FACT: You can return any item within 30 days for a full refund.
  2. TYPE 0: The context specifically excludes electronics from the refund policy, meaning they cannot be returned for a full refund. CORRECTED_FACT: Electronic

---
## Part 2 — Relevancy Detector

**Use case:** Q&A systems, customer service, chatbots.

The relevancy detector checks if the answer **addresses** the user's question — even if the facts in the answer are accurate, they might be irrelevant to what was asked.

Fact scores: `1` = relevant, `0` = not relevant

In [6]:
def check_relevancy(question: str, llm_answer: str) -> dict:
    payload = {"question": question, "llm_answer": llm_answer}
    response = requests.post(RELEVANCY_URL, json=payload, headers=HEADERS)
    response.raise_for_status()
    return response.json()


def display_relevancy(result: dict) -> None:
    score = result.get("summary", {}).get("relevancy_score", 0)
    details = result.get("details", {})
    atomic_facts = details.get("atomic_facts", [])
    relevancy_list = details.get("relevancy_list", [])
    relevancy_response = _parse_nested(details.get("relevancy_response", {}))
    chain_of_thought = relevancy_response.get("chain_of_thought", [])

    # Pad chain_of_thought in case it's shorter than atomic_facts
    cot_padded = list(chain_of_thought) + [""] * max(0, len(atomic_facts) - len(chain_of_thought))

    status = "PASS" if score >= 0.5 else ("WARN" if score >= 0.4 else "FAIL")
    score_color = "green" if score >= 0.5 else ("yellow" if score >= 0.4 else "red")
    print(f"Relevancy Score : {_c(f'{score:.2f} ({score*100:.1f}%)  [{status}]', score_color)}")

    if atomic_facts:
        print(f"\nAtomic Facts ({len(atomic_facts)}):")
        for i, (fact, fs, reasoning) in enumerate(zip(atomic_facts, relevancy_list, cot_padded), 1):
            color = "green" if fs == 1 else "red"
            label = "RELEVANT    " if fs == 1 else "NOT RELEVANT"
            print(f"  {i}. {_c(f'[{label}]', color)} {fact}")

    if chain_of_thought:
        print("\nReasoning:")
        for i, r in enumerate(chain_of_thought, 1):
            color = "green" if r.upper().startswith("RELEVANT") else "red"
            print(f"  {i}. {_c(r, color)}")

In [7]:
# Example 2a: Highly relevant answer
question = "What is C.V. Raman known for?"
answer_relevant = "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering."

print("Question  :", question)
print("LLM Answer:", answer_relevant)
print()

result = check_relevancy(question, answer_relevant)
display_relevancy(result)

Question  : What is C.V. Raman known for?
LLM Answer: C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering.

Relevancy Score : 1.00 (100.0%)  [PASS]

Atomic Facts (2):
  1. [RELEVANT    ] C.V. Raman won the Nobel Prize for Physics in 1930.
  2. [RELEVANT    ] C.V. Raman won the Nobel Prize for his work on light scattering.

Reasoning:
  1. RELEVANT: Winning the Nobel Prize is a significant achievement and is relevant to what C.V. Raman is known for.
  2. RELEVANT: His work on light scattering is a key aspect of what he is known for.


In [8]:
# Example 2b: Tangentially related — correct facts, but misses the question
answer_tangential = (
    "India has produced many great scientists over the decades. "
    "The Indian scientific community has contributed to fields ranging "
    "from mathematics to space exploration."
)

print("Question  :", question)
print("LLM Answer:", answer_tangential)
print()

result = check_relevancy(question, answer_tangential)
display_relevancy(result)

Question  : What is C.V. Raman known for?
LLM Answer: India has produced many great scientists over the decades. The Indian scientific community has contributed to fields ranging from mathematics to space exploration.

Relevancy Score : 0.00 (0.0%)  [FAIL]

Atomic Facts (3):
  1. [NOT RELEVANT] India has produced many great scientists over the decades.
  2. [NOT RELEVANT] The Indian scientific community has contributed to the field of mathematics.
  3. [NOT RELEVANT] The Indian scientific community has contributed to the field of space exploration.

Reasoning:
  1. NOT RELEVANT: General statement about Indian scientists, not specific to C.V. Raman.
  2. NOT RELEVANT: Contribution to mathematics is not related to C.V. Raman's known achievements.
  3. NOT RELEVANT: Contribution to space exploration is not related to C.V. Raman's known achievements.


In [9]:
# Example 2c: Evasive customer-service non-answer
question_cs = "What is the interest rate on your savings account?"
answer_evasive = (
    "We offer a wide range of financial products tailored to your needs. "
    "Our team of experts is always ready to help you find the right solution."
)

print("Question  :", question_cs)
print("LLM Answer:", answer_evasive)
print()

result = check_relevancy(question_cs, answer_evasive)
display_relevancy(result)

Question  : What is the interest rate on your savings account?
LLM Answer: We offer a wide range of financial products tailored to your needs. Our team of experts is always ready to help you find the right solution.

Relevancy Score : 0.00 (0.0%)  [FAIL]

Atomic Facts (4):
  1. [NOT RELEVANT] We offer a wide range of financial products.
  2. [NOT RELEVANT] Our financial products are tailored to your needs.
  3. [NOT RELEVANT] Our team of experts is always ready to help you.
  4. [NOT RELEVANT] Our team of experts helps you find the right solution.

Reasoning:
  1. NOT RELEVANT: General information about financial products does not specify interest rates.
  2. NOT RELEVANT: Tailored financial products do not specify interest rates.
  3. NOT RELEVANT: Expert assistance does not specify interest rates.
  4. NOT RELEVANT: Expert assistance does not specify interest rates.


---
## Part 3 — Combined RAG Pipeline

In a real RAG system both detectors run together:

```
User Question → Retrieve Docs → LLM Generate → Adherence Check
                                              → Relevancy Check
                                              → Approve / Reject
```

We run them **concurrently** to keep latency low.

In [10]:
def evaluate_rag_response(
    question: str,
    context: str,
    llm_answer: str,
    adherence_threshold: float = 0.5,
    relevancy_threshold: float = 0.5,
) -> dict:
    """Run both checks concurrently and return a combined decision."""
    with ThreadPoolExecutor(max_workers=2) as executor:
        fut_adh = executor.submit(check_adherence, llm_answer, context)
        fut_rel = executor.submit(check_relevancy, question, llm_answer)
        adh_result = fut_adh.result()
        rel_result = fut_rel.result()

    adh_score = adh_result.get("summary", {}).get("adherence_score", 0)
    rel_score = rel_result.get("summary", {}).get("relevancy_score", 0)

    approved = adh_score >= adherence_threshold and rel_score >= relevancy_threshold
    return {
        "adherence_score": adh_score,
        "relevancy_score": rel_score,
        "approved": approved,
    }


def display_rag_result(r: dict) -> None:
    print(f"Adherence : {r['adherence_score']:.2f}")
    print(f"Relevancy : {r['relevancy_score']:.2f}")
    print(f"Decision  : {'✅ APPROVED' if r['approved'] else '❌ REJECTED'}")

In [11]:
# Scenario A: Good RAG response
q = "What is C.V. Raman known for?"
ctx = "C.V. Raman won the Nobel Prize for Physics in 1930 for his work on light scattering."
ans = "C.V. Raman is famous for discovering the Raman Effect (light scattering), which won him the Nobel Prize in Physics in 1930."

print("[Scenario A — Good Response]")
print(f"Q: {q}")
print(f"A: {ans}")
print()
result = evaluate_rag_response(q, ctx, ans)
display_rag_result(result)

[Scenario A — Good Response]
Q: What is C.V. Raman known for?
A: C.V. Raman is famous for discovering the Raman Effect (light scattering), which won him the Nobel Prize in Physics in 1930.

Adherence : 1.00
Relevancy : 0.67
Decision  : ✅ APPROVED


In [12]:
# Scenario B: Hallucinated answer — should fail adherence
ans_hallucinated = "C.V. Raman won the Nobel Prize in 1930 for his research on nuclear physics and atomic structure."

print("[Scenario B — Hallucinated Answer]")
print(f"Q: {q}")
print(f"A: {ans_hallucinated}")
print()
result = evaluate_rag_response(q, ctx, ans_hallucinated)
display_rag_result(result)

[Scenario B — Hallucinated Answer]
Q: What is C.V. Raman known for?
A: C.V. Raman won the Nobel Prize in 1930 for his research on nuclear physics and atomic structure.

Adherence : 0.33
Relevancy : 0.33
Decision  : ❌ REJECTED


In [13]:
# Scenario C: Off-topic response — should fail relevancy
q_pw = "How do I reset my password?"
ctx_pw = "Users can reset their password by clicking 'Forgot Password' on the login page."
ans_offtopic = "Our platform uses state-of-the-art AES-256 encryption to protect all user data."

print("[Scenario C — Off-Topic Response]")
print(f"Q: {q_pw}")
print(f"A: {ans_offtopic}")
print()
result = evaluate_rag_response(q_pw, ctx_pw, ans_offtopic)
display_rag_result(result)

[Scenario C — Off-Topic Response]
Q: How do I reset my password?
A: Our platform uses state-of-the-art AES-256 encryption to protect all user data.

Adherence : 0.00
Relevancy : 0.00
Decision  : ❌ REJECTED


---
## Part 4 — Try Your Own

Edit the cell below and test your own LLM responses.

In [14]:
# ── Customize these values ──────────────────────────────────────────────
MY_QUESTION = "What are the symptoms of type 2 diabetes?"

MY_CONTEXT = """
Type 2 diabetes symptoms often develop slowly and can include increased thirst,
frequent urination, fatigue, blurred vision, slow-healing sores, and frequent
infections. Some people with type 2 diabetes have no symptoms at all.
"""

MY_LLM_ANSWER = """
Common symptoms of type 2 diabetes include increased thirst, frequent urination,
fatigue, and blurred vision. Some people may have no symptoms initially.
"""
# ────────────────────────────────────────────────────────────────────────

print("Question  :", MY_QUESTION)
print("Context   :", MY_CONTEXT.strip())
print("LLM Answer:", MY_LLM_ANSWER.strip())
print()

result = evaluate_rag_response(MY_QUESTION, MY_CONTEXT, MY_LLM_ANSWER)
display_rag_result(result)

# Show detailed adherence breakdown
print("\n--- Adherence Detail ---")
adh_detail = check_adherence(MY_LLM_ANSWER, MY_CONTEXT)
display_adherence(adh_detail)

print("\n--- Relevancy Detail ---")
rel_detail = check_relevancy(MY_QUESTION, MY_LLM_ANSWER)
display_relevancy(rel_detail)

Question  : What are the symptoms of type 2 diabetes?
Context   : Type 2 diabetes symptoms often develop slowly and can include increased thirst,
frequent urination, fatigue, blurred vision, slow-healing sores, and frequent
infections. Some people with type 2 diabetes have no symptoms at all.
LLM Answer: Common symptoms of type 2 diabetes include increased thirst, frequent urination,
fatigue, and blurred vision. Some people may have no symptoms initially.

Adherence : 1.00
Relevancy : 1.00
Decision  : ✅ APPROVED

--- Adherence Detail ---
Adherence Score : 1.00 (100.0%)  [PASS]

Atomic Facts (5):
  1. [TYPE 2 — SUPPORTED  ] Increased thirst is a common symptom of type 2 diabetes.
  2. [TYPE 2 — SUPPORTED  ] Frequent urination is a common symptom of type 2 diabetes.
  3. [TYPE 2 — SUPPORTED  ] Fatigue is a common symptom of type 2 diabetes.
  4. [TYPE 2 — SUPPORTED  ] Blurred vision is a common symptom of type 2 diabetes.
  5. [TYPE 2 — SUPPORTED  ] Some people with type 2 diabetes may h

---
## Quick Reference

### Adherence API
```python
POST https://api.enkryptai.com/guardrails/adherence
Headers: { "apikey": "<your-key>", "Content-Type": "application/json" }
Body:    { "llm_answer": "...", "context": "..." }
```
Key response fields:
- `summary.adherence_score` — overall score (0–1)
- `details.atomic_facts` — extracted facts
- `details.adherence_list` — per-fact scores (`2`=supported, `0`=unsupported)
- `details.adherence_response.chain_of_thought` — reasoning

### Relevancy API
```python
POST https://api.enkryptai.com/guardrails/relevancy
Headers: { "apikey": "<your-key>", "Content-Type": "application/json" }
Body:    { "question": "...", "llm_answer": "..." }
```
Key response fields:
- `summary.relevancy_score` — overall score (0–1)
- `details.atomic_facts` — extracted facts
- `details.relevancy_list` — per-fact scores (`1`=relevant, `0`=not relevant)
- `details.relevancy_response.chain_of_thought` — reasoning

### Recommended thresholds
| Score | Meaning |
|---|---|
| ≥ 0.5 | Pass — safe to show to user |
| 0.4–0.5 | Warn — review before showing |
| < 0.4 | Fail — regenerate or flag for human review |

---
## Part 5 — Candidate Summarization Accuracy

**Use case:** Recruiting / talent matching platforms.

A recruiting platform ingests multiple raw data chunks about a candidate, a job posting, and the hiring company, then asks an LLM to produce a concise 50–100 word candidate summary. The **adherence detector** verifies that every claim in the summary is actually supported by the source material — catching fabricated credentials, wrong salary figures, or misattributed company details before they reach a recruiter.

> The context below is intentionally messy and unstructured (as it would arrive from a real data pipeline). The summary contains a few deliberate errors so you can see the detector flag them.

In [5]:
# ── Candidate Summarization Use Case ────────────────────────────────────────
#
# Three raw, lightly-formatted context chunks as they would arrive from a
# real data pipeline (resume parser, ATS export, company profile scrape).

chunk_resume = """
CANDIDATE: Jordan Lee
email: j.lee@email.com  |  ph: 555-0182

EXPERIENCE (7 yrs total):
  -- TechFlow Inc  (2020–present)  |  Lead Engineer
     Python / AWS. Built real-time data pipelines; managed 4-person eng team.
  -- DataBridge Co  (2017–2020)  |  Backend Developer
     Java + Postgres. Core contributor to inventory management system.

EDUCATION:  B.S. Computer Science, University of Texas at Austin, 2017

SKILLS: Python, Java, AWS, PostgreSQL, Docker, REST APIs
CERTIFICATIONS: AWS Solutions Architect – Associate (2022)
"""

chunk_job_posting = """
ROLE: Senior Software Engineer @ Nexus Systems
LOC:  Austin, TX  (hybrid — 3 days onsite required)
SALARY RANGE: $140,000 – $165,000

REQUIREMENTS:
  - Min 5 yrs software engineering experience
  - Strong Python and cloud (AWS preferred)
  - Data pipeline / ETL experience strongly preferred
  - Prior team leadership a plus
START DATE: ASAP / Q1 2026
"""

chunk_company = """
Nexus Systems  —  Company Overview
Founded: 2015  |  Series B ($42 M raised)  |  ~200 employees
HQ: Austin, TX
Core business: logistics automation software for mid-market retailers
Culture: fast-paced, engineering-driven; remote-friendly but prefers Austin locals
Recent highlights: launched NexusRoute 2.0 (Nov 2024); expanded to Canadian market (Q3 2024)
"""

# Combine all three chunks into one context string
candidate_context = "\n---\n".join([chunk_resume, chunk_job_posting, chunk_company])

# ── LLM-generated candidate summary — contains 3 deliberate errors ────────────
#   Error 1: "5 years of experience"   (Jordan actually has 7 years)
#   Error 2: "Series A company"        (Nexus Systems is Series B)
#   Error 3: "salary up to $160,000"   (actual top of range is $165,000)

candidate_summary = (
    "Jordan Lee is a strong match for the Senior Software Engineer role at Nexus Systems. "
    "With 5 years of software engineering experience, including hands-on Python and AWS expertise "
    "and a proven track record leading engineering teams, Jordan meets the core technical requirements. "
    "Jordan holds a B.S. in Computer Science from UT Austin and earned an AWS Solutions Architect "
    "certification in 2022. Nexus Systems is a Series A logistics automation company based in Austin "
    "offering a hybrid role with a salary up to $160,000."
)

print("=" * 70)
print("CANDIDATE SUMMARIZATION — ADHERENCE CHECK")
print("=" * 70)
print("\n[Context — Chunk 1: Resume]")
print(chunk_resume.strip())
print("\n[Context — Chunk 2: Job Posting]")
print(chunk_job_posting.strip())
print("\n[Context — Chunk 3: Company Profile]")
print(chunk_company.strip())
print("\n[LLM-Generated Summary]")
print(candidate_summary)
print()

# Run adherence check — does the summary stay grounded in the three chunks?
candidate_adherence_result = check_adherence(candidate_summary, candidate_context)
display_adherence(candidate_adherence_result)

CANDIDATE SUMMARIZATION — ADHERENCE CHECK

[Context — Chunk 1: Resume]
CANDIDATE: Jordan Lee
email: j.lee@email.com  |  ph: 555-0182

EXPERIENCE (7 yrs total):
  -- TechFlow Inc  (2020–present)  |  Lead Engineer
     Python / AWS. Built real-time data pipelines; managed 4-person eng team.
  -- DataBridge Co  (2017–2020)  |  Backend Developer
     Java + Postgres. Core contributor to inventory management system.

EDUCATION:  B.S. Computer Science, University of Texas at Austin, 2017

SKILLS: Python, Java, AWS, PostgreSQL, Docker, REST APIs
CERTIFICATIONS: AWS Solutions Architect – Associate (2022)

[Context — Chunk 2: Job Posting]
ROLE: Senior Software Engineer @ Nexus Systems
LOC:  Austin, TX  (hybrid — 3 days onsite required)
SALARY RANGE: $140,000 – $165,000

REQUIREMENTS:
  - Min 5 yrs software engineering experience
  - Strong Python and cloud (AWS preferred)
  - Data pipeline / ETL experience strongly preferred
  - Prior team leadership a plus
START DATE: ASAP / Q1 2026

[Context 

### What next? — Correcting the Summary with an LLM

The adherence detector has identified which facts in the summary are unsupported or incorrect. The next step is to feed those findings back into an LLM so it can produce a **corrected, fully-grounded summary** automatically.

The prompt below gives the model explicit knowledge of the detector's scoring system:

| Score | Type | Meaning | Action |
|---|---|---|---|
| `2` | TYPE 2 | Fact directly supported by context | Keep as-is |
| `1` | TYPE 1 | Inferred / not explicitly stated | Rephrase or remove |
| `0` | TYPE 0 | Incorrect — contradicts the context | Replace with the `CORRECTED_FACT` provided by the detector |

The model receives the original context chunks, the flawed summary, and the per-fact detector output, then rewrites the summary to be accurate.

In [6]:
import openai

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_KEY_HERE")
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client ready:", bool(OPENAI_API_KEY and OPENAI_API_KEY != "YOUR_OPENAI_KEY_HERE"))


def build_flagged_issues(adherence_result: dict, colored: bool = False) -> str:
    """
    Convert the adherence detector output into a structured list of issues
    for the correction prompt (and optionally for colored display).

    Scoring key:
      2 = TYPE 2 — directly supported, no action needed
      1 = TYPE 1 — inferred/not explicitly stated, rephrase or remove
      0 = TYPE 0 — incorrect, use the CORRECTED_FACT provided by the detector
    """
    details = adherence_result.get("details", {})
    atomic_facts = details.get("atomic_facts", [])
    adherence_list = details.get("adherence_list", [])
    adherence_response = _parse_nested(details.get("adherence_response", {}))
    chain_of_thought = adherence_response.get("chain_of_thought", [])

    issues = []
    for fact, score, reasoning in zip(atomic_facts, adherence_list, chain_of_thought):
        if score == 2:
            continue  # fully supported — skip
        if "TYPE 0" in reasoning:
            color, label = "red", "TYPE 0 — INCORRECT (must fix using CORRECTED_FACT below)"
        else:
            color, label = "yellow", "TYPE 1 — INFERRED (rephrase to stick to what the context explicitly states)"

        claim_line  = f'  Claim : "{fact}"'
        status_line = f'  Status: {_c(label, color) if colored else label}'
        detail_line = f'  Detail: {_c(reasoning, color) if colored else reasoning}'
        issues.append(f"{claim_line}\n{status_line}\n{detail_line}")

    if not issues:
        return _c("None — all facts are supported.", "green") if colored else "None — all facts are supported."
    return "\n\n".join(issues)


def correct_summary(
    original_summary: str,
    context: str,
    adherence_result: dict,
    target_words: str = "50–100",
) -> str:
    """Call GPT to rewrite the summary with all detector-flagged errors fixed."""

    flagged_issues = build_flagged_issues(adherence_result)

    system_prompt = f"""You are a precision fact-correction assistant for a recruiting platform.
Your task is to rewrite a candidate summary so that every claim is explicitly grounded in the provided source context.

The Enkrypt AI adherence detector has already analysed the summary and flagged problematic claims.
It uses the following scoring system:

  TYPE 2 — The fact is directly and explicitly supported by the context.
            → Keep it unchanged.

  TYPE 1 — The fact is an inference or editorial judgement not directly stated in the context.
            → Remove it or rephrase it using only language the context supports.

  TYPE 0 — The fact is factually incorrect and contradicts the context.
            The detector provides a CORRECTED_FACT — you must use it.

Output only the corrected summary ({target_words} words). Do not add explanations, headers, or commentary."""

    user_prompt = f"""SOURCE CONTEXT (three raw data chunks):
{context}

ORIGINAL SUMMARY (contains errors):
{original_summary}

FLAGGED ISSUES FROM ENKRYPT AI ADHERENCE DETECTOR:
{flagged_issues}

Write the corrected summary now."""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()


# ── Run the correction ────────────────────────────────────────────────────────
print("=" * 70)
print("STEP 1 — ORIGINAL SUMMARY (with errors)")
print("=" * 70)
print(candidate_summary)

print("\n" + "=" * 70)
print("STEP 2 — FLAGGED ISSUES FROM ENKRYPT AI ADHERENCE DETECTOR")
print("=" * 70)
print(build_flagged_issues(candidate_adherence_result, colored=True))

print("\n" + "=" * 70)
print("STEP 3 — CORRECTED SUMMARY (via GPT-4o-mini)")
print("=" * 70)
corrected = correct_summary(candidate_summary, candidate_context, candidate_adherence_result)
print(corrected)

# Optional: re-run the adherence check on the corrected summary to confirm improvement
print("\n" + "=" * 70)
print("STEP 4 — RE-CHECK ADHERENCE ON CORRECTED SUMMARY")
print("=" * 70)
corrected_result = check_adherence(corrected, candidate_context)
display_adherence(corrected_result)

OpenAI client ready: True
STEP 1 — ORIGINAL SUMMARY (with errors)
Jordan Lee is a strong match for the Senior Software Engineer role at Nexus Systems. With 5 years of software engineering experience, including hands-on Python and AWS expertise and a proven track record leading engineering teams, Jordan meets the core technical requirements. Jordan holds a B.S. in Computer Science from UT Austin and earned an AWS Solutions Architect certification in 2022. Nexus Systems is a Series A logistics automation company based in Austin offering a hybrid role with a salary up to $160,000.

STEP 2 — FLAGGED ISSUES FROM ENKRYPT AI ADHERENCE DETECTOR
  Claim : "Jordan Lee is a strong match for the Senior Software Engineer role at Nexus Systems."
  Status: TYPE 1 — INFERRED (rephrase to stick to what the context explicitly states)
  Detail: TYPE 1: Jordan Lee is a strong match for the Senior Software Engineer role at Nexus Systems because he meets the requirements such as having over 5 years of exper